**Persiapan SparkSession**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Tugas6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

**A. EXTRACT** *(bobot 15%)*

Baca ketiga sumber data (`tugas6_transaksi.csv`, `tugas6_produk.json`, `tugas6_ulasan.csv`) menjadi tiga Spark DataFrame terpisah.
Tampilkan jumlah baris dan `printSchema()` masing-masing.

In [ ]:
# Membaca data transaksi
df_transaksi = spark.read.csv(
    "tugas6_transaksi.csv",
    header=True,
    inferSchema=True
)

# Membaca data produk
df_produk = spark.read.json("tugas6_produk.json")

# Membaca data ulasan
df_ulasan = spark.read.csv(
    "tugas6_ulasan.csv",
    header=True,
    inferSchema=True
)

print("Data transaksi:")
df_transaksi.printSchema()

print("Data produk:")
df_produk.printSchema()

print("Data ulasan:")
df_ulasan.printSchema()

In [ ]:
# pastikan datanya terbaca
print("Jumlah data transaksi:", df_transaksi.count())
print("Jumlah data produk:", df_produk.count())
print("Jumlah data ulasan:", df_ulasan.count())

**B. TRANSFORM — Penggabungan** *(bobot 25%)*

Gabungkan ketiga DataFrame menjadi satu (`order_id` sebagai kunci ke ulasan, `product_id` sebagai kunci ke produk). Gunakan 
**`salah satu join yang tepat`** dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan 
**`salah satu join yang tepat`** dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid).
    Tambahkan kolom `total_pendapatan` (`unit_terjual x harga`).

1. Gabungkan transaksi dengan ulasan

In [ ]:
df_transaksi_ulasan = df_transaksi.join(
    df_ulasan,
    on="order_id",
    how="left"
)

df_transaksi_ulasan.show(10)

print("Jumlah transaksi setelah join ulasan:", df_transaksi_ulasan.count())

2. Gabungkan dengan data produk

In [ ]:
df_etl = df_transaksi_ulasan.join(
    df_produk,
    on="product_id",
    how="inner"
)

df_etl.show(10)

df_etl.printSchema()

3. Tambahkan total pendapatan Sesuai soal: total_pendapatan = unit_terjual x harga

In [ ]:
df_etl = df_etl.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga")
)

df_etl.show(10)

**C. TRANSFORM — Penanganan Data Kosong & Pengayaan** *(bobot 20%)*

- Transaksi tanpa ulasan akan memiliki `rating` bernilai kosong (`null`) setelah `salah satu join yang tepat` — isi nilai kosong tersebut dengan angka **0** menggunakan `salah satu function`, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
- Tambahkan kolom `ada_ulasan` bernilai `True`/`False` (tidak boleh diisi manual satu satu)`, **sebelum** langkah `na.fill()` di atas).


In [ ]:
from pyspark.sql.functions import when, col

# Kode Anda yang sebelumnya di bawah ini
df_etl = df_etl.fillna({"rating": 0})
df_etl = df_etl.withColumn(
    "ada_ulasan",
    when(col("rating") > 0, True).otherwise(False)
)

print("Jumlah rating yang masih kosong:",
      df_etl.filter(col("rating").isNull()).count())
         
    

**D. LOAD** *(bobot 25%)*

Simpan hasil akhir ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori`, ke path `/user/[username]/tugas6/hasil_etl`.
Verifikasi dengan `hdfs dfs -ls -R`, lalu baca kembali dan tampilkan `count()`-nya sebagai bukti data tersimpan utuh.

In [ ]:
hdfs_path = "hdfs://localhost:9000/user/vey/tugas6/hasil_etl"

df_etl.write \
    .mode("overwrite") \
    .partitionBy("kategori") \
    .parquet(hdfs_path)

print("Data berhasil disimpan ke HDFS!")

!hdfs dfs -ls -R /user/vey/tugas6/hasil_etl

In [ ]:
df_final = spark.read.parquet(
    "hdfs://localhost:9000/user/vey/tugas6/hasil_etl"
)

print("Jumlah baris hasil akhir:", df_final.count())

df_final.show(10)

**E. Insight Akhir** *(bobot 15%)*

Dari data hasil ETL, tampilkan (menggunakan DataFrame API **atau** Spark SQL, bebas memilih): kategori produk mana yang memiliki **persentase transaksi dengan ulasan** (`ada_ulasan = True`) **paling rendah**? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?

In [ ]:
from pyspark.sql.functions import col, count, sum, round

# Menghitung jumlah transaksi dan transaksi yang memiliki ulasan
insight = df_final.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    sum(when(col("ada_ulasan") == True, 1).otherwise(0)).alias("transaksi_dengan_ulasan")
)

# Menghitung persentase transaksi yang memiliki ulasan
insight = insight.withColumn(
    "persentase_ulasan",
    round(col("transaksi_dengan_ulasan") / col("total_transaksi") * 100, 2)
)

insight.show()

**explore**

In [ ]:
#  Deteksi Produk dengan Rating Sempurna (Extract & Filter Lanjutan)
from pyspark.sql.functions import col

# Filter hanya transaksi yang mendapatkan rating sempurna (Bintang 5)
df_rating_sempurna = df_etl.filter(col("rating") == 5)

print("=== EKSPLORASI 1: SAMPEL PRODUK DENGAN RATING BINTANG 5 ===")
df_rating_sempurna.select("order_id", "nama_produk", "kategori", "harga", "rating") \
                  .orderBy(col("harga").desc()) \
                  .show(5)

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum

# Meringkas jumlah unit terjual berdasarkan kelompok kategori
df_eksplorasi_unit = df_etl.groupBy("kategori").agg(
    spark_sum("unit_terjual").alias("total_unit_laku")
)

print("=== JUMLAH UNIT TERJUAL PER KATEGORI PRODUK ===")
df_eksplorasi_unit.orderBy(col("total_unit_laku").desc()).show()

In [ ]:
spark.stop()
print("SparkSession ditutup.")